In [ ]:
import jax, jax.numpy as jnp
from jax.scipy.linalg import solve_triangular


# Model: data = response_function(xi) + noise
N = 10

prior_cov_matrix = jnp.eye(N)
data_cov_matrix = jnp.eye(N)
data = jnp.zeros(N)
inv_data_cov_matrix = jnp.linalg.inv(data_cov_matrix)
inv_prior_cov_matrix = jnp.linalg.inv(prior_cov_matrix)
tolerance = 1e-5
def response_function(xi):
    return xi
def negative_logdensity(xi):
    res = response_function(xi) - data
    return res.T @ inv_data_cov_matrix @ res + xi.T @ inv_prior_cov_matrix @ xi
def fisher_information_metric(xi):
    J = jax.jacobian(response_function)(xi)
    return J.T @ inv_data_cov_matrix @ J + inv_prior_cov_matrix


def make_H(metric, negative_logdensity):        # metric: q -> (n,n) SPD;  log_post: q -> scalar
    def H(q, p):
        G = metric(q)
        L = jnp.linalg.cholesky(G)                    # G = L Lᵀ, lower
        logdet = 2.0 * jnp.sum(jnp.log(jnp.diag(L)))  # stable log det
        y = solve_triangular(L, p, lower=True)        # y = L⁻¹ p
        quad = jnp.dot(y, y)                          # pᵀ G⁻¹ p = ‖y‖²
        return negative_logdensity(q) + 0.5*logdet + 0.5*quad
    return H

H     = make_H(fisher_information_metric, negative_logdensity)
dH_dq = jax.jit(jax.grad(H, argnums=0))
dH_dp = jax.jit(jax.grad(H, argnums=1))   # returns G⁻¹ p

def one_leapfrog_step(position, momentum):
    

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
